In [1]:
import copy

import numpy as np
import pandas as pd
import anndata as ad

In [2]:

import torch
import torch.nn as nn
import torch.optim as optim

### Load Data

In [3]:
input_dir = "/Users/joshuayang/Desktop/KB/data"
adata_train = ad.read_h5ad(input_dir+'/Shaffer_cancer/shaffer_train.h5ad')
adata_test = ad.read_h5ad(input_dir+'/Shaffer_cancer/shaffer_test.h5ad')


train_labels = adata_train.obs["clone_id"].to_numpy()
test_labels = adata_test.obs["clone_id"].to_numpy()

embed_dir = input_dir + "/feat_LCL_2025/shaffer_cancer/feat_shaffer_lambda01_unlab5_bs100"
X_train  = np.load(embed_dir+'/train_base_embed.npy')
X_test = np.load(embed_dir+'/test_base_embed.npy')


print(adata_train.shape, adata_test.shape)
print(X_train.shape, X_test.shape)

(20656, 2000) (2368, 2000)
(20656, 64) (2368, 64)


In [7]:
adata_train.obs["clone_id"].value_counts()

clone_id
349    936
389    932
527    604
22     493
447    405
      ... 
18       5
379      5
304      5
197      5
464      5
Name: count, Length: 558, dtype: int64

In [4]:
adata_train.obs

,orig.ident,nCount_RNA,nFeature_RNA,nCount_lineage,nFeature_lineage,percent.mt,percent.rb,S.Score,G2M.Score,Phase,RNA_snn_res.0.5,seurat_clusters,RNA_snn_res.0.4,OG_condition,RNA_snn_res.0.3,Lineage,keep,clone_id,OG_condition_name
cistodabtram_CCCGGAAAGCAACTCT-1,SeuratProject,14914.0,4051,29.0,3,6.108355,9.145769,-0.003093,-0.154415,0,0,8,10,3,8,349,1,349,cistodabtram
cistodabtram_CATTCATAGCTAATGA-1,SeuratProject,4243.0,2031,24.0,3,7.612538,7.164742,-0.071747,-0.145342,0,8,7,7,3,7,349,1,349,cistodabtram
cistodabtram_CCTCAGTTCCTCTTTC-1,SeuratProject,24000.0,5311,43.0,5,12.800000,7.000000,-0.059770,0.227976,1,0,8,10,3,8,349,1,349,cistodabtram
cis_ACGATGTGTCGCGTTG-1,SeuratProject,10600.0,3288,8.0,3,11.943396,8.500000,-0.025121,0.107649,1,7,3,8,0,3,349,1,349,cis
cistodabtram_TTCATGTAGGGAGATA-1,SeuratProject,7200.0,2709,27.0,1,6.833333,7.611111,0.517406,-0.041500,2,1,8,10,3,8,349,1,349,cistodabtram
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
cocl2_TACGCTCAGCATGCAG-1,SeuratProject,6434.0,2193,33.0,2,6.310227,12.449487,-0.141095,-0.199297,0,8,7,7,4,7,464,1,464,cocl2
cocl2tocis_AAACCCACAATCCTAG-1,SeuratProject,7333.0,2816,49.0,8,12.491477,7.623074,0.076776,-0.172086,2,0,1,2,5,1,464,1,464,cocl2tocis
cocl2tocis_AGTGACTCAACCCGCA-1,SeuratProject,8566.0,2979,37.0,12,15.958440,8.452020,0.398226,-0.233381,2,5,3,6,5,3,464,1,464,cocl2tocis
cocl2tocis_GTGGGAATCTACAGGT-1,SeuratProject,5345.0,2324,70.0,9,12.029935,7.988775,-0.061381,0.571929,1,6,3,8,5,3,464,1,464,cocl2tocis


In [5]:
# -----------------------
# 0) Utilities for Shaffer condition parsing
# -----------------------
def add_shaffer_condition_columns(
    adata,
    condition_key="OG_condition_name",
):
    """
    Adds:
      - is_future: True if condition is a 2-treatment condition (contains 'to')
      - first_treatment
      - second_treatment
      - branch_id = clone_id + '__' + first_treatment

    Assumes condition names like:
      cis, cocl2, dabtram
      cistocis, cistococl2, cistodabtram, ...
    """
    cond = adata.obs[condition_key].astype(str)

    is_future = cond.str.contains("to", regex=False)

    first_treatment = []
    second_treatment = []

    for x in cond:
        if "to" in x:
            a, b = x.split("to", 1)
            first_treatment.append(a)
            second_treatment.append(b)
        else:
            first_treatment.append(x)
            second_treatment.append("none")

    adata.obs["is_future"] = is_future.to_numpy()
    adata.obs["first_treatment"] = np.array(first_treatment, dtype=object)
    adata.obs["second_treatment"] = np.array(second_treatment, dtype=object)

    # branch_id will be created later once clone_id exists
    return adata


def add_branch_id(
    adata,
    lineage_key="clone_id",
    first_treatment_key="first_treatment",
):
    adata.obs["branch_id"] = (
        adata.obs[lineage_key].astype(str) + "__" + adata.obs[first_treatment_key].astype(str)
    )
    return adata


# -----------------------
# 1) Optional filter: keep branches with enough future cells
# -----------------------
def filter_by_branch_future_size_shaffer(
    adata,
    X,
    lineage_key="clone_id",
    condition_key="OG_condition_name",
    min_future_cells=10,
):
    """
    Keep all cells whose branch (clone_id + first_treatment) has >= min_future_cells
    among FUTURE cells only (2-treatment cells), computed within this split only.
    """
    adata = adata.copy()
    add_shaffer_condition_columns(adata, condition_key=condition_key)
    add_branch_id(adata, lineage_key=lineage_key)

    future_mask = adata.obs["is_future"].to_numpy()
    future_branch_counts = adata.obs.loc[future_mask, "branch_id"].value_counts()

    keep_branches = set(
        future_branch_counts[future_branch_counts >= int(min_future_cells)].index
    )

    keep_mask = adata.obs["branch_id"].isin(keep_branches).to_numpy()

    return adata[keep_mask].copy(), X[keep_mask]


# -----------------------
# 2) Build early inputs + future 9-way composition targets
# -----------------------
def build_targets_from_future_shaffer(
    X,
    adata,
    lineage_key="clone_id",
    condition_key="OG_condition_name",
    future_categories=None,
    alpha_smooth=1e-3,
    drop_missing_future=True,
):
    """
    Early inputs:
      cells with single-treatment conditions (no 'to')

    Target:
      for each early cell, use its branch_id = (clone_id, first_treatment)
      and build a probability vector over 9 future conditions using ONLY future cells
      from the same branch_id.

    Example:
      early cell: clone 349 under 'cis'
      target = composition of future cells from branch (349, cis)
      across categories:
        cistocis, cistococl2, cistodabtram, cocl2tocis, ... dabtramtodabtram
      In practice only the 3 categories beginning with 'cis' should be nonzero.
    """
    adata = adata.copy()
    add_shaffer_condition_columns(adata, condition_key=condition_key)
    add_branch_id(adata, lineage_key=lineage_key)

    # Identify early / future
    is_future = adata.obs["is_future"].to_numpy()
    is_early = ~is_future

    adata_future = adata[is_future].copy()
    adata_early = adata[is_early].copy()
    X_early = X[is_early]

    # Determine category order
    if future_categories is None:
        future_categories = sorted(adata_future.obs[condition_key].astype(str).unique().tolist())
    future_categories = list(future_categories)
    C = len(future_categories)

    # branch_id -> probability vector over 9 future conditions
    branch_to_probs = {}
    for branch_id, df in adata_future.obs.groupby("branch_id"):
        counts = np.array(
            [(df[condition_key].astype(str) == cat).sum() for cat in future_categories],
            dtype=float
        )
        counts = counts + alpha_smooth
        probs = counts / counts.sum()
        branch_to_probs[branch_id] = probs

    # Assign targets to early cells
    early_branch_ids = adata_early.obs["branch_id"].astype(str).to_numpy()

    y_prob = np.zeros((X_early.shape[0], C), dtype=float)
    keep = np.ones(X_early.shape[0], dtype=bool)

    for i, bid in enumerate(early_branch_ids):
        if bid in branch_to_probs:
            y_prob[i] = branch_to_probs[bid]
        else:
            if drop_missing_future:
                keep[i] = False
            else:
                y_prob[i] = np.ones(C) / C

    X_early = X_early[keep]
    y_prob = y_prob[keep]
    y_prob = y_prob / y_prob.sum(axis=1, keepdims=True)

    return (
        torch.tensor(X_early, dtype=torch.float32),
        torch.tensor(y_prob, dtype=torch.float32),
        future_categories,
    )


# -----------------------
# 3) Linear decoder
# -----------------------
class LinearSoftmax(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc(x)  # logits


# -----------------------
# 4) Train with early stopping
# -----------------------
def train_kl_earlystop(
    model,
    X_train,
    y_train,
    lr=5e-3,
    weight_decay=1e-4,
    max_epochs=5000,
    batch_size=256,
    val_frac=0.2,
    patience=150,
    min_delta=1e-5,
    seed=42,
    device=None,
    print_every=50,
    verbose=True,
):
    if device is None:
        if torch.cuda.is_available():
            device = "cuda"
        elif torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"

    model = model.to(device)
    X_train = X_train.to(device)
    y_train = y_train.to(device)

    n = X_train.shape[0]
    g = torch.Generator(device="cpu").manual_seed(seed)
    perm = torch.randperm(n, generator=g)

    n_val = int(round(val_frac * n))
    val_idx = perm[:n_val]
    tr_idx = perm[n_val:]

    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_val, y_val = X_train[val_idx], y_train[val_idx]

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.KLDivLoss(reduction="batchmean")

    best_val = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = -1
    bad_epochs = 0

    @torch.no_grad()
    def eval_loss(Xe, ye):
        model.eval()
        log_probs = torch.log_softmax(model(Xe), dim=1)
        return criterion(log_probs, ye).item()

    for ep in range(1, max_epochs + 1):
        model.train()

        perm_tr = torch.randperm(X_tr.shape[0], device=device)
        Xs = X_tr[perm_tr]
        ys = y_tr[perm_tr]

        total = 0.0
        for start in range(0, Xs.shape[0], batch_size):
            xb = Xs[start:start + batch_size]
            yb = ys[start:start + batch_size]

            logits = model(xb)
            log_probs = torch.log_softmax(logits, dim=1)
            loss = criterion(log_probs, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total += loss.item() * xb.shape[0]

        train_loss = total / Xs.shape[0]
        val_loss = eval_loss(X_val, y_val)

        improved = (best_val - val_loss) > min_delta
        if improved:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = ep
            bad_epochs = 0
        else:
            bad_epochs += 1

        if verbose and (ep == 1 or ep % print_every == 0):
            print(
                f"Epoch {ep}/{max_epochs} | train={train_loss:.6f} | val={val_loss:.6f} "
                f"| best_val={best_val:.6f} (ep {best_epoch}) | bad={bad_epochs}/{patience} | device={device}"
            )

        if bad_epochs >= patience:
            if verbose:
                print(f"Early stopping at epoch {ep}. Best val={best_val:.6f} at epoch {best_epoch}.")
            break

    model.load_state_dict(best_state)
    return model, {"best_epoch": best_epoch, "best_val": best_val, "device": device}


@torch.no_grad()
def eval_kl(model, X, y):
    model.eval()
    device = next(model.parameters()).device
    X = X.to(device)
    y = y.to(device)
    criterion = nn.KLDivLoss(reduction="batchmean")
    log_probs = torch.log_softmax(model(X), dim=1)
    return criterion(log_probs, y).item()


# -----------------------
# 5) Full Shaffer KL experiment
# -----------------------
def run_one_threshold_experiment_shaffer(
    adata_train, X_train,
    adata_test, X_test,
    lineage_threshold=10,
    lineage_key="clone_id",
    condition_key="OG_condition_name",
    future_categories=None,
    alpha_smooth=1e-3,
    device="mps",
    seed=42,
    lr=5e-3,
    weight_decay=1e-4,
    max_epochs=5000,
    batch_size=256,
    val_frac=0.2,
    patience=150,
    min_delta=1e-5,
    print_every=50,
):
    # 1) filter branches by number of future cells, separately in train/test
    ad_tr_f, X_tr_f = filter_by_branch_future_size_shaffer(
        adata_train, X_train,
        lineage_key=lineage_key,
        condition_key=condition_key,
        min_future_cells=lineage_threshold,
    )
    ad_te_f, X_te_f = filter_by_branch_future_size_shaffer(
        adata_test, X_test,
        lineage_key=lineage_key,
        condition_key=condition_key,
        min_future_cells=lineage_threshold,
    )

    # 2) fix the 9 target categories using TRAIN split only
    if future_categories is None:
        tmp = ad_tr_f.obs[condition_key].astype(str)
        future_categories = sorted(tmp[tmp.str.contains("to", regex=False)].unique().tolist())

    # 3) build early -> future composition pairs
    X_tr2, y_tr, future_categories = build_targets_from_future_shaffer(
        X_tr_f,
        ad_tr_f,
        lineage_key=lineage_key,
        condition_key=condition_key,
        future_categories=future_categories,
        alpha_smooth=alpha_smooth,
    )

    X_te2, y_te, _ = build_targets_from_future_shaffer(
        X_te_f,
        ad_te_f,
        lineage_key=lineage_key,
        condition_key=condition_key,
        future_categories=future_categories,
        alpha_smooth=alpha_smooth,
    )

    print("Future categories:", future_categories)
    print("Train early cells used:", X_tr2.shape[0])
    print("Test early cells used:", X_te2.shape[0])

    # 4) train decoder
    model = LinearSoftmax(input_size=X_tr2.shape[1], output_size=len(future_categories))
    model, hist = train_kl_earlystop(
        model,
        X_tr2, y_tr,
        lr=lr,
        weight_decay=weight_decay,
        max_epochs=max_epochs,
        batch_size=batch_size,
        val_frac=val_frac,
        patience=patience,
        min_delta=min_delta,
        seed=seed,
        device=device,
        print_every=print_every,
        verbose=True,
    )

    # 5) evaluate
    kl_train = eval_kl(model, X_tr2, y_tr)
    kl_test = eval_kl(model, X_te2, y_te)

    summary = {
        "min_future_cells": lineage_threshold,
        "future_categories": future_categories,
        "train_cells_total_after_filter": ad_tr_f.n_obs,
        "test_cells_total_after_filter": ad_te_f.n_obs,
        "train_time1_cells_used": X_tr2.shape[0],
        "test_time1_cells_used": X_te2.shape[0],
        "best_epoch": hist["best_epoch"],
        "val_KL_best": hist["best_val"],
        "train_KL": kl_train,
        "test_KL": kl_test,
        "device": hist["device"],
    }

    print(summary)
    return model, summary

In [6]:
model, summary = run_one_threshold_experiment_shaffer(
    adata_train, X_train,
    adata_test, X_test,
    lineage_threshold=10,      # tune this
    lineage_key="clone_id",
    condition_key="OG_condition_name",
    device="mps",              # or "cuda"
    seed=42,
    max_epochs=10000,
    patience=150,
    print_every=50,
)

Future categories: ['cistocis', 'cistococl2', 'cistodabtram', 'cocl2tocis', 'cocl2tococl2', 'cocl2todabtram', 'dabtramtocis', 'dabtramtococl2', 'dabtramtodabtram']
Train early cells used: 1565
Test early cells used: 57
Epoch 1/10000 | train=1.485180 | val=1.270246 | best_val=1.270246 (ep 1) | bad=0/150 | device=mps
Epoch 50/10000 | train=0.123108 | val=0.126350 | best_val=0.126350 (ep 50) | bad=0/150 | device=mps
Epoch 100/10000 | train=0.079816 | val=0.090707 | best_val=0.090707 (ep 100) | bad=0/150 | device=mps
Epoch 150/10000 | train=0.063660 | val=0.078098 | best_val=0.078098 (ep 150) | bad=0/150 | device=mps
Epoch 200/10000 | train=0.055500 | val=0.072432 | best_val=0.072432 (ep 200) | bad=0/150 | device=mps
Epoch 250/10000 | train=0.050641 | val=0.070046 | best_val=0.069991 (ep 249) | bad=1/150 | device=mps
Epoch 300/10000 | train=0.047360 | val=0.068950 | best_val=0.068866 (ep 296) | bad=4/150 | device=mps
Epoch 350/10000 | train=0.045083 | val=0.068459 | best_val=0.068459 (ep 3